In [7]:
# import necessary libraries
import numpy as np
import pandas as pd

In [8]:
# loading the cleaned datasets
customers=pd.read_parquet("../data/02_cleaned/customers_clean.parquet")
geolocation=pd.read_parquet("../data/02_cleaned/geolocation_clean.parquet")
order_items=pd.read_parquet("../data/02_cleaned/order_items_clean.parquet")
order_payments=pd.read_parquet("../data/02_cleaned/order_payments_clean.parquet")
order_reviews=pd.read_parquet("../data/02_cleaned/order_reviews_clean.parquet")
orders=pd.read_parquet("../data/02_cleaned/orders_clean.parquet")
products=pd.read_parquet("../data/02_cleaned/products_clean.parquet")
sellers=pd.read_parquet("../data/02_cleaned/sellers_clean.parquet")

In [9]:
# displaying the columns of each dataframe
display(customers.columns)
display(geolocation.columns)
display(order_items.columns)
display(order_payments.columns)
display(order_reviews.columns)
display(orders.columns)
display(products.columns)
display(sellers.columns)

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='object')

Index(['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state'],
      dtype='object')

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='object')

Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='object')

Index(['review_id', 'order_id', 'review_score', 'no_review_message',
       'review_message', 'review_creation_date', 'review_answer_timestamp'],
      dtype='object')

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')

Index(['product_id', 'product_category_ename', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='object')

Index(['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state'], dtype='object')

In [10]:
# chain merging
df_final = orders.merge(order_payments , on='order_id', how='left') \
.merge(order_reviews , on='order_id', how='left') \
.merge(customers , on='customer_id', how='left') \
.merge(order_items , on='order_id', how='left') \
.merge(products , on='product_id', how='left') \
.merge(sellers , on='seller_id', how='left')
# '\' is line continuation character used to split long lines for better readability

In [11]:
# the zip code prefix in geolocation is not unique for every lat lng pair
geo_clean = geolocation.groupby('geolocation_zip_code_prefix').agg({'geolocation_lat':'mean', 'geolocation_lng':'mean'}).reset_index()
# final merge
df_final = df_final.merge(geo_clean , left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')

In [12]:
# saving the final merged dataframe to a csv file
df_final.to_parquet("../data/03_merged/ecommerce_merged.parquet")